# Real-world Data Wrangling

In this project, you will apply the skills you acquired in the course to gather and wrangle real-world data with two datasets of your choice.

You will retrieve and extract the data, assess the data programmatically and visually, accross elements of data quality and structure, and implement a cleaning strategy for the data. You will then store the updated data into your selected database/data store, combine the data, and answer a research question with the datasets.

Throughout the process, you are expected to:

1. Explain your decisions towards methods used for gathering, assessing, cleaning, storing, and answering the research question
2. Write code comments so your code is more readable

## 1. Gather data

In this section, you will extract data using two different data gathering methods and combine the data. Use at least two different types of data-gathering methods.

### **1.1.** Problem Statement
In 2-4 sentences, explain the kind of problem you want to look at and the datasets you will be wrangling for this project.

This project investigates which aircraft manufacturers are more frequently associated with fatal aviation accidents in the United States. I also explore whether the year an aircraft was manufactured correlates with its involvement in fatal accidents. To answer these questions, I will wrangle data from the NTSB fatal aviation accident database (available on Kaggle) and the FAA aircraft registry, which contains detailed information about registered aircraft including manufacturer and year of manufacture.

### **1.2.** Gather at least two datasets using two different data gathering methods

List of data gathering methods:

- Download data manually
- Programmatically downloading files
- Gather data by accessing APIs
- Gather and extract data from HTML files using BeautifulSoup
- Extract data from a SQL database

Each dataset must have at least two variables, and have greater than 500 data samples within each dataset.

For each dataset, briefly describe why you picked the dataset and the gathering method (2-3 full sentences), including the names and significance of the variables in the dataset. Show your work (e.g., if using an API to download the data, please include a snippet of your code). 

Load the dataset programmtically into this notebook.

In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### Dataset 1 - NTSB Fatal Aviation Accidents

**Type:** CSV  
**Method:** Programmatic download via Kaggle API (kagglehub)

I chose this dataset because it contains records of all fatal aviation accidents reported to the NTSB from January 2010 through February 2025, providing over 5,000 records to analyze. The data was gathered programmatically using the `kagglehub` Python library, which downloads the dataset directly from Kaggle's servers. This approach is reproducible and ensures the latest version of the dataset is always retrieved.

**Variables:**
- **N** — The aircraft tail number (N-number), which uniquely identifies each aircraft registered with the FAA
- **EventDate** — The date and time when the accident occurred, stored as an ISO 8601 string
- **State** — The U.S. state (or location) where the accident took place

In [ ]:
# Download the NTSB fatal aviation accident dataset from Kaggle
path = kagglehub.dataset_download("sainin/fatal-aviation-accidents-jan2010-feb2025")
accident_data = pd.read_csv(os.path.join(path, "a8f0c8d2-d8a5-4420-91b0-9033d2064f6fAviationData.csv"))

# Save the full raw data before any modifications
accident_data.to_csv("data-wrangling/raw-accident-data.csv", index=False)

# Keep only the columns relevant to our analysis
accident_data = accident_data[["N", "EventDate", "State"]]

accident_data.head()

#### Dataset 2 - FAA Aircraft Registry

**Type:** CSV  
**Method:** Manual download from the FAA website

I selected this dataset because it contains the official FAA registry of all aircraft registered in the United States, which allows me to link accident records to specific aircraft details. The data was manually downloaded from the FAA's aircraft registration database website as a CSV file. This dataset provides over 300,000 records of registered aircraft.

**Variables:**
- **N-NUMBER** — The aircraft's registration number, which corresponds to the tail number used in NTSB reports
- **YEAR MFR** — The year the aircraft was manufactured, used to analyze whether aircraft age correlates with accident involvement
- **MFR MDL CODE** — A code that links to the aircraft model reference table, allowing us to look up the manufacturer and model name

In [ ]:
# Load the FAA aircraft registry data (manually downloaded from FAA website)
registry_data = pd.read_csv("data-wrangling/raw-aircraft-registry.csv")

# Keep only the columns relevant to our analysis
registry_data = registry_data[["N-NUMBER", "YEAR MFR", "MFR MDL CODE"]]

registry_data.head()

#### Dataset 3 - FAA Aircraft Registry - Models

**Type:** CSV  
**Method:** Manual download from the FAA website

I chose this supplementary dataset because it maps manufacturer/model codes to human-readable manufacturer and model names. This reference table was manually downloaded alongside the aircraft registry from the FAA website. Without this dataset, the aircraft registry only contains numeric codes for manufacturers, making analysis of accident rates by manufacturer impossible.

**Variables:**
- **CODE** — The unique manufacturer/model code that corresponds to MFR MDL CODE in the aircraft registry
- **MFR** — The name of the aircraft manufacturer (e.g., Cessna, Piper, Beech)
- **MODEL** — The specific model designation of the aircraft (e.g., PA-32-300, SR22T)

In [ ]:
# Load the FAA aircraft model reference table (manually downloaded from FAA website)
model_data = pd.read_csv("data-wrangling/aircraft-models.csv")

# Keep only the columns relevant to our analysis
model_data = model_data[["CODE", "MFR", "MODEL"]]

#### Merge Aircraft and Model

In [ ]:
# Merge aircraft registry with model reference table to get manufacturer and model names
aircraft_data = registry_data.merge(model_data, left_on="MFR MDL CODE", right_on="CODE", how="left")

aircraft_data.head()

Optional data storing step: You may save your raw dataset files to the local data store before moving to the next step.

In [ ]:
# Save working copies of the datasets after column selection but before cleaning
# Raw data is preserved separately:
#   - raw-accident-data.csv (full Kaggle download saved in cell above)
#   - aircraft-registry.csv (original FAA registry download)
#   - aircraft-models.csv (original FAA model reference download)
accident_data.to_csv("data-wrangling/accident-data.csv", index=False)
aircraft_data.to_csv("data-wrangling/aircraft-data.csv", index=False)

## 2. Assess data

Assess the data according to data quality and tidiness metrics using the report below.

List **two** data quality issues and **two** tidiness issues. Assess each data issue visually **and** programmatically, then briefly describe the issue you find.  **Make sure you include justifications for the methods you use for the assessment.**

### Quality Issue 1:

In [ ]:
# Visual assessment: inspect the first few rows for obvious issues
accident_data.head()

# Programmatic assessment: identify all exact duplicate rows
accident_data[accident_data.duplicated(keep=False)]

**Issue:** There are duplicate rows in the accident dataset. The row for aircraft D-ECHF on 2024-09-07 appears twice with identical values across all columns.

**Assessment justification:** I used `.head()` to visually scan for obvious anomalies and `.duplicated(keep=False)` to programmatically flag all rows that have identical values across every column. This method checks the **Uniqueness** quality pillar by identifying exact duplicate records that would inflate accident counts if left uncleaned.

### Quality Issue 2:

In [ ]:
# Visual assessment: inspect all unique state values sorted alphabetically
print(accident_data["State"].value_counts().sort_index())

# Programmatic assessment: filter for known non-state values
invalid_states = ["Atlantic Ocean", "Caribbean Sea", "Gulf of Mexico"]
accident_data[accident_data["State"].isin(invalid_states)]

**Issue:** The State column contains values that are bodies of water (Atlantic Ocean, Caribbean Sea, Gulf of Mexico) rather than U.S. states, affecting 16 records.

**Assessment justification:** I used `.value_counts().sort_index()` to visually scan all unique state values in alphabetical order, which made the non-state entries easy to spot. I then used `.isin()` to programmatically filter and count the affected rows. This addresses the **Validity** quality pillar, since these values don't belong in a column intended for U.S. states.

### Tidiness Issue 1:

In [ ]:
# Programmatic assessment: find rows where the N column contains commas (multiple aircraft)
accident_data[accident_data["N"].str.contains(",", na=False)]

**Issue:** When multiple aircraft are involved in a single accident, all tail numbers are stored in a single cell separated by commas (e.g., "N709PS, UNREG"). This violates the tidiness rule that each observation should form its own row, since each aircraft's involvement is a separate observation.

**Assessment justification:** I used `.str.contains(",")` to programmatically identify all rows where the N column contains multiple values. The output above also serves as a visual confirmation of the issue. This revealed 96 rows with multiple tail numbers that need to be split into separate rows so each aircraft can be individually analyzed and joined with the registry data.

### Tidiness Issue 2: 

In [ ]:
# Visual assessment: inspect the format of EventDate values
print(accident_data["EventDate"].head(10))

# Programmatic assessment: check the data type and string length consistency
print(f"\nData type: {accident_data['EventDate'].dtype}")
print(f"Sample value contains 'T' separator: {'T' in str(accident_data['EventDate'].iloc[0])}")
print(f"String lengths: {accident_data['EventDate'].str.len().value_counts().to_dict()}")

**Issue:** The EventDate column encodes two distinct variables — date and time — in a single ISO 8601 string (e.g., "2025-02-06T16:20:00Z"). This violates the tidiness principle that each variable should form its own column. Since our analysis only requires the date, we will extract just the date component.

**Assessment justification:** I used `.head()` to visually confirm the combined datetime format and checked `.dtype` programmatically to verify the column is stored as a generic object (string) rather than a proper datetime type. The consistent string length of 20 characters confirms all values follow the same ISO 8601 format with both date and time encoded together.

### Quality Issue 3:

In [ ]:
# Programmatic assessment: check data types and non-null counts
aircraft_data.info()

In [ ]:
# Visual assessment: check value distribution to spot blank/whitespace entries
aircraft_data["YEAR MFR"].value_counts()

In [ ]:
# Visual assessment: bar chart showing the most common YEAR MFR values including blanks
aircraft_data["YEAR MFR"].value_counts().head(15).plot(kind="bar")

In [ ]:
# Programmatic assessment: wrap values in quotes to reveal hidden whitespace
aircraft_data["YEAR MFR"].apply(lambda x: f"'{x}'").head()

**Issue:** The FAA registry data uses fixed-width string formatting, so all values are padded with trailing spaces. Missing values appear as whitespace-only strings (e.g., "    ") rather than proper NaN values, which means pandas cannot correctly identify them as missing data. The YEAR MFR column shows 64,080 blank entries that `.info()` incorrectly reports as non-null.

**Assessment justification:** I used `.info()` to programmatically check data types and non-null counts, which revealed all 309,586 rows appear non-null — masking truly missing values. I then used `.value_counts()` and a bar chart to visually spot the blank entries, and a lambda function wrapping values in quotes to programmatically confirm the whitespace padding. This addresses the **Validity** quality pillar, since whitespace-only strings are not valid year values.

### **Quality Issue 4: N-Numbers not consistent**

In [ ]:
# Visual assessment: compare the format of N-numbers between datasets
print("Aircraft registry N-NUMBER format (no 'N' prefix):")
print(aircraft_data["N-NUMBER"].head(2))

print("\nAccident data N format (has 'N' prefix, includes foreign aircraft):")
print(accident_data["N"].head(2))

# Programmatic assessment: count accident records with non-US tail numbers
non_us = accident_data[~accident_data["N"].str.startswith("N", na=True)]
print(f"\nAccident records with non-US tail numbers: {len(non_us)}")
print(f"Registry records (all US-registered): {len(aircraft_data)}")

**Issue:** The N-numbers are formatted differently between the two datasets — the FAA registry omits the "N" prefix (e.g., "100") while the NTSB accident data includes it (e.g., "N321BA"). Additionally, the accident data contains foreign aircraft with non-U.S. registration prefixes (e.g., "PK-LUV") that cannot be matched to the FAA registry.

**Assessment justification:** I used `.head()` to visually compare the N-number formats between datasets, confirming the prefix mismatch. I then programmatically filtered for records not starting with "N" to count foreign aircraft entries. This addresses the **Consistency** quality pillar, ensuring the join key has a matching format across both datasets before merging.

## 3. Clean data
Clean the data to solve the 4 issues corresponding to data quality and tidiness found in the assessing step. **Make sure you include justifications for your cleaning decisions.**

After the cleaning for each issue, please use **either** the visually or programatical method to validate the cleaning was succesful.

At this stage, you are also expected to remove variables that are unnecessary for your analysis and combine your datasets. Depending on your datasets, you may choose to perform variable combination and elimination before or after the cleaning stage. Your dataset must have **at least** 4 variables after combining the data.

In [ ]:
# Make copies of the datasets to ensure the raw dataframes are not modified
cleaned_accident_data = accident_data.copy()
cleaned_aircraft_data = aircraft_data.copy()

### **Quality Issue 1: Duplicate Data**

In [ ]:
# Remove exact duplicate rows, keeping the first occurrence
cleaned_accident_data = cleaned_accident_data.drop_duplicates()

In [ ]:
# Validate: confirm no duplicate rows remain
cleaned_accident_data[cleaned_accident_data.duplicated(keep=False)]

**Justification:** I used `.drop_duplicates()` to remove exact duplicate rows because these identical records would artificially inflate accident counts in our analysis. I chose to keep the first occurrence (default behavior) since all columns are identical in the duplicated rows, making the choice of which to keep arbitrary. The validation above confirms zero duplicates remain.

### **Quality Issue 2: Bodies of water in state column**

In [ ]:
# Remove rows where State is a body of water rather than a U.S. state
cleaned_accident_data = cleaned_accident_data[~cleaned_accident_data["State"].isin(invalid_states)]

In [ ]:
# Validate: confirm no body-of-water entries remain
cleaned_accident_data[cleaned_accident_data["State"].isin(invalid_states)]

**Justification:** I chose to remove rows with ocean/sea locations rather than mapping them to the nearest state because the actual crash location over water cannot be reliably assigned to a specific state. Since our research question focuses on manufacturer and year trends rather than geographic analysis, dropping these 16 records (less than 0.3% of the data) has minimal impact on our analysis while maintaining data accuracy.

### **Tidiness Issue 1: Multiple tailnumbers**

In [ ]:
# Split comma-separated tail numbers into individual rows so each aircraft gets its own record
cleaned_accident_data = cleaned_accident_data.assign(
    N=cleaned_accident_data["N"].str.split(",")
).explode("N")

# Remove leading/trailing whitespace from tail numbers after splitting
cleaned_accident_data["N"] = cleaned_accident_data["N"].str.strip()

In [ ]:
# Validate: confirm no comma-separated values remain in the N column
cleaned_accident_data[cleaned_accident_data["N"].str.contains(",", na=False)]

**Justification:** I used `.str.split(",")` followed by `.explode()` to create separate rows for each aircraft involved in a multi-aircraft accident. This approach preserves the shared accident details (date, state) for each aircraft while enabling individual aircraft lookups in the FAA registry. An alternative would have been to keep only the first tail number, but that would discard data about other aircraft involved and undercount certain manufacturers' accident involvement. The validation confirms no comma-separated values remain.

### **Tidiness Issue 2: Date and time in the same column**

In [ ]:
# Convert the EventDate string to a date-only value, removing the time component
cleaned_accident_data["EventDate"] = pd.to_datetime(cleaned_accident_data["EventDate"]).dt.date

In [ ]:
# Validate: confirm EventDate now contains only date values without timestamps
cleaned_accident_data["EventDate"].head(10)

**Justification:** I used `pd.to_datetime()` to parse the ISO 8601 string and `.dt.date` to extract only the date portion. Since our research question focuses on manufacturer and year trends rather than time-of-day analysis, the time component is unnecessary. Converting to a proper date type also enables correct chronological sorting and date-based operations. The validation shows the cleaned values contain only dates without timestamps.

### **Quality Issue 3: Spaces**

In [ ]:
# Strip leading/trailing whitespace from all string columns
cleaned_aircraft_data = cleaned_aircraft_data.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# Convert empty strings to NaN so pandas correctly identifies missing values
cleaned_aircraft_data = cleaned_aircraft_data.replace("", np.nan)

In [ ]:
# Validate: confirm no empty strings remain and .info() correctly reports missing values
empty_counts = (cleaned_aircraft_data == "").sum()
empty_counts[empty_counts > 0]

print(empty_counts)

cleaned_aircraft_data.info()

**Justification:** I used `.str.strip()` to remove the fixed-width padding because the whitespace prevents accurate string matching (e.g., when joining on N-NUMBER) and makes value comparisons unreliable. I then converted empty strings to `NaN` using `.replace()` so that pandas correctly identifies the 64,080 missing YEAR MFR records, enabling proper use of `.dropna()`, `.fillna()`, and `.info()` for downstream analysis. The validation confirms zero empty strings remain and `.info()` now correctly reports 245,506 non-null values for YEAR MFR.

### **Quality Issue 4: N-Numbers not consistent and data contains foreign aircraft and unregistered aircraft**

In [ ]:
# Add "N" prefix to registry N-numbers to match the accident data format
cleaned_aircraft_data["N-NUMBER"] = "N" + cleaned_aircraft_data["N-NUMBER"]

# Remove accident records with non-US (foreign) tail numbers since they won't match the FAA registry
cleaned_accident_data = cleaned_accident_data[cleaned_accident_data["N"].str.startswith("N", na=False)]

# Remove accident records with missing tail numbers
cleaned_accident_data = cleaned_accident_data[cleaned_accident_data["N"].notna()]

In [ ]:
# Validate: confirm N-numbers now have the "N" prefix
cleaned_aircraft_data.head()

**Justification:** I chose to add the "N" prefix to the registry data (rather than stripping it from accident data) because the "N" prefix is the standard FAA registration format and keeps the identifiers human-readable. Foreign aircraft were removed from the accident data because they cannot be matched to the FAA registry and are outside the scope of our U.S.-focused analysis. Records with missing tail numbers were also removed since they cannot be joined to any aircraft record. The validation shows the registry N-numbers now include the "N" prefix, matching the accident data format.

### **Remove unnecessary variables and combine datasets**

Depending on the datasets, you can also peform the combination before the cleaning steps.

In [ ]:
# Preview both datasets before merging
print(cleaned_accident_data.head())
print(cleaned_aircraft_data.head())

# Merge accident data with aircraft data using tail number as the join key
combined_data = cleaned_accident_data.merge(cleaned_aircraft_data, left_on="N", right_on="N-NUMBER", how="left")

combined_data.head(10)

## 4. Update your data store
Update your local database/data store with the cleaned data, following best practices for storing your cleaned data:

- Must maintain different instances / versions of data (raw and cleaned data)
- Must name the dataset files informatively
- Ensure both the raw and cleaned data is saved to your database/data store

In [ ]:
# Save the cleaned datasets to preserve both raw and cleaned versions
# Raw data: raw-accident-data.csv, aircraft-registry.csv, aircraft-models.csv
# Cleaned data: files saved below
cleaned_accident_data.to_csv("data-wrangling/cleaned-accident-data.csv", index=False)
cleaned_aircraft_data.to_csv("data-wrangling/cleaned-aircraft-data.csv", index=False)
combined_data.to_csv("data-wrangling/combined-data.csv", index=False)

## 5. Answer the research question

### **5.1:** Define and answer the research question 
Going back to the problem statement in step 1, use the cleaned data to answer the question you raised. Produce **at least** two visualizations using the cleaned data and explain how they help you answer the question.

*Research question:* What manufacturers of aircraft are more involved in fatal accidents than other ones? Does year manufactured have something to do with aircraft fatal accidents?

In [ ]:
# Count accidents and aircraft by manufacturer
accidents = combined_data[combined_data["MFR"].notna()].groupby("MFR").size()
fleet = cleaned_aircraft_data.groupby("MFR").size()

# Calculate rate per 1000 aircraft (only manufacturers with 100+ aircraft)
rates = (accidents / fleet * 1000).dropna()
rates = rates[fleet >= 100].sort_values(ascending=False).head(15)

# Plot
rates.plot(kind="barh", figsize=(10, 6))
plt.xlabel("Fatal Accidents per 1,000 Aircraft")
plt.ylabel("Manufacturer")
plt.title("Fatal Accident Rate by Manufacturer")
plt.gca().invert_yaxis()
plt.tight_layout()

*Answer to research question:* This chart reveals that certain smaller manufacturers like Experimental and kit-built aircraft categories have higher fatal accident rates per 1,000 aircraft than major manufacturers like Cessna and Piper.

In [ ]:
# Get year data and filter out bad values
year_data = combined_data[combined_data["YEAR MFR"].notna()].copy()
year_data["YEAR MFR"] = pd.to_numeric(year_data["YEAR MFR"])
year_data = year_data[year_data["YEAR MFR"] > 1900]

aircraft_years = cleaned_aircraft_data[cleaned_aircraft_data["YEAR MFR"].notna()].copy()
aircraft_years["YEAR MFR"] = pd.to_numeric(aircraft_years["YEAR MFR"])
aircraft_years = aircraft_years[aircraft_years["YEAR MFR"] > 1900]

# Group by decade
year_data["Decade"] = (year_data["YEAR MFR"] // 10) * 10
aircraft_years["Decade"] = (aircraft_years["YEAR MFR"] // 10) * 10

# Calculate rate
accidents_by_decade = year_data.groupby("Decade").size()
fleet_by_decade = aircraft_years.groupby("Decade").size()
rate_by_decade = accidents_by_decade / fleet_by_decade * 1000

# Plot
rate_by_decade.plot(kind="bar", figsize=(10, 6))
plt.xlabel("Decade Manufactured")
plt.ylabel("Fatal Accidents per 1,000 Aircraft")
plt.title("Fatal Accident Rate by Decade Manufactured")
plt.tight_layout()

In [ ]:
result = pd.DataFrame({                                                                                                                                                                                                                                                                                              
    "Accidents": accidents_by_decade,                                                                                                                                                                                                                                                                                
    "Fleet Size": fleet_by_decade,                                                                                                                                                                                                                                                                                   
    "Rate per 1000": rate_by_decade.round(2)                                                                                                                                                                                                                                                                         
})                                                                                                                                                                                                                                                                                                                   

result

*Answer to research question:* The data shows that newer aircraft have higher fatal accident rates per 1,000 registered aircraft. This is likely because the FAA registry includes all     
 registered aircraft, but many older planes are no longer actively flying (stored, grounded, or in museums). This suggests that year manufacturered a lone is not a good predictor of safety.

### **5.2:** Reflection
In 2-4 sentences, if you had more time to complete the project, what actions would you take? For example, which data quality and structural issues would you look into further, and what research questions would you further explore?

*Answer:* With more time, I would investigate the specific models within each manufacturer to identify if certain models have higher accident rates. Since date manufacturered isn't a good predictor of safety, I would also account for flight hours rather than just registered aircraft counts, as some aircraft fly more frequently than others.